# CPSC 4830 – Final Exam (Amazon Review Assistant with GPT)

**Duration:** 3 Hours  
**Total Marks:** 100  

This notebook is your **exam paper**. All answers must be written in this notebook.

---

## IMPORTANT INSTRUCTIONS

- You **must use Google Colab** on the **exam room PCs**.  
  - **Personal laptops are not allowed.** No exceptions.
- You **may NOT** use ChatGPT, Copilot, Claude, or any other external AI assistant.
- You **may** use:
  - The course materials and examples on **D2L**  
  - The official **Hugging Face `datasets` documentation**: <https://huggingface.co/docs/datasets>  
  - The official **OpenAI API documentation**: <https://platform.openai.com/docs/api-reference>  
- Do **not** hard-code your real OpenAI API key into the notebook you submit.  
  - Use environment variables / Colab secrets, and remove the key before uploading.

**Submission:**  
- When you are done, make sure all cells have been run.  
- Download the `.ipynb` from Colab and upload it to D2L under the Final Exam submission.

---


## Marking Overview (Rubric Summary)

- **Part 1 – Load, Explore, and Sample Reviews:** 25 marks  
- **Part 2 – GPT Summarization of Reviews:** 35 marks  
- **Part 3 – GPT Sentiment Classification of Summaries:** 25 marks  
- **Part 4 – Mini “Review Assistant” Chatbot:** 15 marks  

Total = **100 marks**.

Detailed mark breakdown is included in each part below.


---

## Part 1 – Load, Explore, and Sample Reviews (25 marks)

We will use the **public `amazon_polarity` dataset** from Hugging Face.

> Dataset: `amazon_polarity`  
> Columns of interest:  
> • `label` – 0 = negative, 1 = positive  
> • `content` – review text

Work on a manageable subset (for example, the first 20,000 rows) so that Colab runs smoothly.

### Part 1.1 – Load a review split (10 marks)

**Goal:** Load the dataset and inspect the first rows.

**What to do:**
- Use `datasets.load_dataset("amazon_polarity", split="train[:20000]")` (or similar).  
- Display the first **10 rows**, showing at least:
  - the **label** column  
  - the **content** column (review text)

**Marks (10):**
- Correct dataset loading (5)  
- Correct display of 10 rows with the required columns (5)

---

### Part 1.2 – Label distribution (10 marks)

**Goal:** Understand how labels are distributed.

**What to do:**
- Compute and print a **frequency table** and **percentage breakdown** of `label` values (0 and 1).  
- Add a **short markdown explanation (2–3 sentences)** commenting on whether the data is balanced or skewed.

**Marks (10):**
- Correct frequency & percentage calculation (6)  
- Clear commentary about balance / skew (4)

---

### Part 1.3 – Balanced sample (5 marks)

**Goal:** Build a balanced subset called `balanced_reviews`.

**What to do:**
- From the loaded split, create a subset of **1,000 reviews** with **exactly 500 label=0 and 500 label=1** (sample with replacement if needed).  
- Show the new label distribution for `balanced_reviews`.

**Marks (5):**
- Balanced sampling logic (3)  
- Correct distribution printout (2)


### Part 1.1 – Starter code for dataset loading (you may modify as needed)

In [149]:
# Install required packages (run once at the top of your notebook in Colab)
!pip install -q datasets pandas

from datasets import load_dataset
import pandas as pd

# TODO: Load a manageable subset of the amazon_polarity dataset.
# Hint: use split="train[:20000]"

dataset = load_dataset(
    "amazon_polarity",
    split="train[:20000]"
)

print(dataset)

df = dataset.to_pandas()
print(df.head(10)[["label", "content"]])  # you may adjust this


Dataset({
    features: ['label', 'title', 'content'],
    num_rows: 20000
})
   label                                            content
0      1  This sound track was beautiful! It paints the ...
1      1  I'm reading a lot of reviews saying that this ...
2      1  This soundtrack is my favorite music of all ti...
3      1  I truly like this soundtrack and I enjoy video...
4      1  If you've played the game, you know how divine...
5      1  I am quite sure any of you actually taking the...
6      0  This is a self-published book, and if you want...
7      1  I loved Whisper of the wicked saints. The stor...
8      1  I just finished reading Whisper of the Wicked ...
9      1  This was a easy to read book that made me want...


In [150]:
# Part 1.2 – Compute label distribution
# TODO: Compute counts and percentages for each label value (0 and 1).

display(df.describe())
# display(df.columns)
# display(df.head(5))
df_count = df["label"].value_counts()
display(df_count)
df_percentage = df["label"].value_counts(normalize=True)*100
display(df_percentage)

,label
count,20000.000000
mean,0.512850
std,0.499847
min,0.000000
25%,0.000000
50%,1.000000
75%,1.000000
max,1.000000


,count
label,
1,10257
0,9743


,proportion
label,
1,51.285
0,48.715


In [151]:
# Part 1.3 – Create balanced_reviews subset
# TODO: Sample exactly 500 rows with label=0 and 500 rows with label=1 (with replacement if needed).
# Store result in a variable named balanced_reviews.

balanced_reviews_0 = df[df["label"] == 0].sample(n=500, replace=True)
balanced_reviews_1 = df[df["label"] == 1].sample(n=500, replace=True)
display(balanced_reviews_0.shape)
display(balanced_reviews_1.shape)
# balanced_reviews = balanced_reviews_1.merge(balanced_reviews_0, how="outer")
# display(balanced_reviews.shape)
# balanced_reviews = balanced_reviews.append(df[df["label"] == 1].sample(n=500, replace=True))
# Combine the two samples vertically
balanced_reviews = pd.concat([balanced_reviews_0, balanced_reviews_1], axis=0)

# Shuffle the combined dataframe to mix labels
balanced_reviews = balanced_reviews.sample(frac=1, random_state=42).reset_index(drop=True)

# Display the shape and label distribution to verify
display(balanced_reviews.shape)
display(balanced_reviews['label'].value_counts())
display(balanced_reviews.head(5))


(500, 3)

(500, 3)

(1000, 3)

,count
label,
1,500
0,500


,label,title,content
0,1,Great for smoothing and repairing sensitive skin,Some days my skin is very oily and other days ...
1,1,It's all based on what you like...,"Personally, I find the reviews on this site to..."
2,1,Fantastic. Up close and personal. Scripturally...,Doesn't get much better than this. What a bles...
3,1,AFX - Smojphace,"I think AFX's take on ""Run the Place Red"" is i..."
4,0,Product didn't match the picture/description,Unless they sent me the wrong product by mista...


_Use the next markdown cell to briefly describe your observations about the label distribution._

**Part 1.2 / 1.3 – Short commentary (2–3 sentences):**  
 I can see in the count that the dataset  when i count by  label (1,2), the presence of label 1 is  a little bigger 10257,vs the label 0 9743,so for a correct analysis we mus try to get balanced over the sampling, also looks like thes dataset similar to  spam where the label can classify the messages can  be calssified as spam or not, we cat be sure at this point.

---

## Part 2 – GPT Summarization of Reviews (35 marks)

In this part, you will use the **OpenAI chat API** to summarize review text.

Use a small, cost-effective model (for example `gpt-4.1-mini` or a model specified by your instructor).  
Refer to the OpenAI API docs: <https://platform.openai.com/docs/api-reference/chat>

We will summarize the `content` column of `balanced_reviews`.

### Part 2.1 – Summarization function (15 marks)

Write a function `summarize_review(text: str) -> str` that:

- Calls the **OpenAI chat completion** API using a **system** + **user** message pattern.  
- Asks the model to produce a **concise, neutral summary** in **30–50 words**, based only on the review text.  
- Returns the summary as a plain string.

**Marks (15):**
- Correct use of OpenAI chat API (8)  
- Clear summarization prompt with word limit (5)  
- Robustness (e.g., basic error handling / stripping whitespace) (2)

---

### Part 2.2 – Apply summarization to reviews (10 marks)

Apply `summarize_review` to (at least) the **first 100 rows** of `balanced_reviews`.

- Create a new column `summary` containing the GPT output.  
- Show a small table with **5 examples** that includes:
  - original `content`  
  - `label`  
  - `summary`

**Marks (10):**
- Successful application to ~100 reviews (6)  
- Clear table of 5 example rows (4)

---

### Part 2.3 – Save summarized data (10 marks)

Create a **Pandas DataFrame** with at least:

- `review_id` (or index)  
- `label`  
- `summary`  

Save it as **`summarized_reviews.csv`**.

**Marks (10):**
- Correct DataFrame construction (5)  
- File saved and path shown (5)


In [152]:
# Part 2.1 – Setup OpenAI client and define summarization function
# Hints:
# - Use environment variables or Colab's secret storage for your API key.
# - Use the chat completions API with a system + user message.

# n this part, you will use the OpenAI chat API to summarize review text.

# Use a small, cost-effective model (for example gpt-4.1-mini or a model specified by your instructor).
# Refer to the OpenAI API docs: https://platform.openai.com/docs/api-reference/chat

# We will summarize the content column of balanced_reviews.

# Part 2.1 – Summarization function (15 marks)
# Write a function summarize_review(text: str) -> str that:

# Calls the OpenAI chat completion API using a system + user message pattern.
# Asks the model to produce a concise, neutral summary in 30–50 words, based only on the review text.
# Returns the summary as a plain string.
# Marks (15):

# Correct use of OpenAI chat API (8)
# Clear summarization prompt with word limit (5)
# Robustness (e.g., basic error handling / stripping whitespace) (2)


# TODO: import and configure OpenAI client
# from openai import OpenAI
# client = OpenAI()

# TODO: define summarize_review(text: str) -> str
import os
import openai


from openai import OpenAI
from google.colab import userdata
apikey=userdata.get('OPENAI_API_KEY')
# print(apikey)
# openai.api_key = apikey

from openai import OpenAI
client = OpenAI(api_key=apikey)
model="gpt-4.1-mini",

def summarize_review(text: str):

  system_message = {
        "role": "system",
        "content": (
            "You are a helpful assistant. "
            "Please provide a concise, neutral summary of the given review text "
            "in 30 to 50 words, based only on the content provided."
        )
    }
  user_message = {
        "role": "user",
        "content": text
    }

  messages=[
    {"role": "developer", "content": "You are a helpful assistant."},
    # {"role" : "developer",  "content": (
    #         "You are a helpful assistant. "
    #         "Please provide a concise, neutral summary of the given review text "
    #         "in 30 to 50 words, based only on the content provided."
    #     )},
    {"role": "user", "content": text}
  ]
  try:
    completion = client.chat.completions.create(
      model="gpt-4.1-mini",
      temperature=0.1,
      max_tokens=250,
      messages=[system_message, user_message]

)


    return completion.choices[0].message.content
  except Exception as e:
    print(e)
    return ""


# question="may i pass  if i solve  3 points of  6 the finals of cspc 4830"
test_function=summarize_review(question)
display(test_function)

'The dataset contains 20,000 reviews with labels, titles, content, summaries, and sentiment labels. Summaries provide concise neutral evaluations of products, movies, or books, highlighting key points such as quality, performance, and user experience. The assistant uses these summaries to answer product-related questions based solely on provided review contexts.'

In [153]:
# Part 2.2 – Apply summarization to at least 100 reviews
# TODO: apply summarize_review to the first 100 rows of balanced_reviews and store in a 'summary' column.
variable_magic_number_view=100
balanced_reviews['summary'] = balanced_reviews[:variable_magic_number_view]['content'].apply(summarize_review)

# Display the DataFrame with summaries
display(balanced_reviews.head(variable_magic_number_view))

# print(response)


,label,title,content,summary
0,1,Great for smoothing and repairing sensitive skin,Some days my skin is very oily and other days ...,The reviewer uses the product a few nights wee...
1,1,It's all based on what you like...,"Personally, I find the reviews on this site to...",The reviewer suggests that opinions on this CD...
2,1,Fantastic. Up close and personal. Scripturally...,Doesn't get much better than this. What a bles...,The reviewer expresses deep appreciation for P...
3,1,AFX - Smojphace,"I think AFX's take on ""Run the Place Red"" is i...","The reviewer finds AFX's ""Run the Place Red"" e..."
4,0,Product didn't match the picture/description,Unless they sent me the wrong product by mista...,The reviewer received a different product than...
...,...,...,...,...
95,0,Why Cripsin? Why?,This was one of the worst movies I've seen in ...,The reviewer found the movie to be one of the ...
96,1,"If you buy, you will have them all....","If you can't make one of the concerts, you can...",This CD offers a great selection of Clay's bes...
97,0,HP has pulled 1022 driver from Website,"Yesterday, HP pulled the Mac OS X driver for t...",HP removed the Mac OS X driver for the 1022 pr...
98,0,"Good book, bad copy.","My problem isn't with the story, it's with thi...",The reviewer appreciates the story but critici...


In [154]:
# Part 2.2 – Show 5 example summaries
# TODO: display 5 rows with [content, label, summary]

display(balanced_reviews.head(5)[["content", "label", "summary"]])



,content,label,summary
0,Some days my skin is very oily and other days ...,1,The reviewer uses the product a few nights wee...
1,"Personally, I find the reviews on this site to...",1,The reviewer suggests that opinions on this CD...
2,Doesn't get much better than this. What a bles...,1,The reviewer expresses deep appreciation for P...
3,"I think AFX's take on ""Run the Place Red"" is i...",1,"The reviewer finds AFX's ""Run the Place Red"" e..."
4,Unless they sent me the wrong product by mista...,0,The reviewer received a different product than...


In [155]:
# Part 2.3 – Save summarized data
# TODO: build a DataFrame with [review_id/index, label, summary] and save to 'summarized_reviews.csv'
balanced_reviews.to_csv('summarized_reviews.csv', index=False)


---

## Part 3 – GPT Sentiment Classification of Summaries (25 marks)

Now you will classify the sentiment of the **summaries**, using GPT again.

### Part 3.1 – Sentiment function (15 marks)

Write a function `classify_sentiment(summary: str) -> str` that:

- Uses the OpenAI chat API with a prompt such as:  
  *"Read this summarized review and classify its overall sentiment as one of: Positive, Neutral, or Negative. Return only the label."*  
- Returns exactly one of: `"Positive"`, `"Neutral"`, `"Negative"`.

Apply this function to all rows in `summarized_reviews` (or at least the same 100 rows), storing the result in a new column `sentiment_label`.

**Marks (15):**
- Correct API usage and prompt (8)  
- Labels restricted to the three options (5)  
- Applied to all relevant rows (2)

---

### Part 3.2 – Compare labels vs sentiment (10 marks)

Create a **cross-tabulation** (e.g., `pd.crosstab`) of dataset `label` vs `sentiment_label` and print it.

Add a short markdown commentary (3–4 sentences) describing:

- Where dataset labels and GPT sentiment appear to agree (e.g., label=1 mostly → Positive).  
- Any mismatches and at least one possible reason.

**Marks (10):**
- Correct cross-tab (6)  
- Sensible commentary on alignment and mismatches (4)


In [156]:
# Part 3.1 – Define classify_sentiment and apply to summaries
# TODO: define classify_sentiment(summary: str) -> str using the chat API
# TODO: apply it to summarized_reviews and add a 'sentiment_label' column

def classify_sentiment(summary: str) -> str:

    prompt = (
        "Read this summarized review and classify its overall sentiment as one of: "
        "Positive, Neutral, or Negative. Return only the label."
        f"\n\nSummary: \"{summary}\""
    )

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that classifies sentiment."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            stop=None,
        )
        label = response.choices[0].message['content'].strip()
        # Ensure label is one of the three allowed
        if label not in {"Positive", "Neutral", "Negative"}:
            label = "Neutral"
        return label
    except Exception as e:
        print(f"Error classifying sentiment: {e}")
        # On error, fallback to Neutral
        return "Neutral"


balanced_reviews=balanced_reviews.sort_values(by='summary', ascending=True)
variable_magic_number_view=10
balanced_reviews['sentiment_label'] = balanced_reviews[:variable_magic_number_view]['summary'].apply(classify_sentiment)

# Display the DataFrame with summaries
display(balanced_reviews.head(variable_magic_number_view))


Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}
Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}
Error classifying sentiment: Error code: 400 - {'error': {'message': "We could not parse the JSON body of yo

,label,title,content,summary,sentiment_label
84,0,Very Dissapointing,From Potters Field is one Patricia Cornwell's ...,"""From Potters Field"" by Patricia Cornwell is c...",Neutral
51,1,serious comedy,FUNNY PEOPLE is somewhat compelling and engagi...,"""Funny People"" is an engaging film about a com...",Neutral
8,1,WOW! A MUST HAVE!,"This album is incredible! It has been too, too...",Clay Aiken's album Measure of a Man is praised...,Neutral
97,0,HP has pulled 1022 driver from Website,"Yesterday, HP pulled the Mac OS X driver for t...",HP removed the Mac OS X driver for the 1022 pr...,Neutral
13,1,Flawless!,Jennifer Love Hewitt shines as actess and huma...,Jennifer Love Hewitt delivers a charming and d...,Neutral
82,0,Snake Oil,The Digital Media Reader may work well with Wi...,The Digital Media Reader (UCS-200) failed to w...,Neutral
54,0,Disappointed...,"We purchased the HD Tivo DVR, which requires o...",The HD Tivo DVR adapter provides poor wireless...,Neutral
77,0,Doesn't Seem to Work with Print Servers,This printer is inexpensive and fast but does ...,The HP1020 printer is affordable and fast but ...,Neutral
17,1,I have heard thier music in the movie Southlander,"This is really neat music, that makes you feel...","The album features enjoyable, feel-good music,...",Neutral
87,1,Perfect for my pickup.,I have a S10 pickup with a topper and my optio...,The bike rack fits well on an S10 pickup with ...,Neutral


In [157]:
# Part 3.2 – label vs sentiment cross-tab
# TODO: create a pd.crosstab of dataset label vs sentiment_label and print the result

crosstab_result = pd.crosstab(balanced_reviews['label'], balanced_reviews['sentiment_label'], rownames=['Original Label'], colnames=['Sentiment Label'])

# Print the crosstab
print(crosstab_result)



Sentiment Label  Neutral
Original Label          
0                      5
1                      5


**Part 3.2 – Short commentary (3–4 sentences):**  
*(Discuss agreement/disagreement between dataset labels and GPT sentiment, and why mismatches might happen.)*

---

## Part 4 – Mini “Review Assistant” Chatbot (15 marks)

You will build a very small **product Q&A assistant** that uses a handful of **summaries** as context.

### Part 4.1 – Prepare context (5 marks)

- Choose a small group of summaries (for example, 5–10 rows from `summarized_reviews`).  
- Combine them into a single **context string** that includes the label and summary for each review.

**Marks (5):**
- Reasonable subset and context construction (5)

---

### Part 4.2 – Implement `ask_review_assistant` (10 marks)

Write a function `ask_review_assistant(question: str) -> str` that:

- Uses the chat API with:
  - A **system message**, for example:  
    *"You are an assistant that answers questions about a product using ONLY the review summaries in the provided context. If the context does not contain enough information, say you don't know."*  
  - A **user message** that includes:
    - The context string (from Part 4.1)  
    - The user’s question

- Returns the model's answer.

Demonstrate the assistant by asking at least **3 different questions** about the product (e.g., durability, common complaints, suitability as a gift).

**Marks (10):**
- Correct system+user prompt design using the summaries as context (6)  
- At least 3 meaningful questions and answers (4)


In [158]:
# Part 4.1 – Build a context from several summaries
# TODO: select 5–10 summarized reviews and build a single context string that includes label + summary.


variable_magic_number_view=10
string_combinedqna= balanced_reviews[:variable_magic_number_view]['summary']

display(string_combinedqna)
context_string = ""
for idx, row in string_combinedqna.items():
    context_string += f"Label {balanced_reviews.loc[idx, 'label']} Summary: {row}\n"

display(context_string)

context_string = str(context_string.strip())
display(context_string)


,summary
84,"""From Potters Field"" by Patricia Cornwell is c..."
51,"""Funny People"" is an engaging film about a com..."
8,Clay Aiken's album Measure of a Man is praised...
97,HP removed the Mac OS X driver for the 1022 pr...
13,Jennifer Love Hewitt delivers a charming and d...
82,The Digital Media Reader (UCS-200) failed to w...
54,The HD Tivo DVR adapter provides poor wireless...
77,The HP1020 printer is affordable and fast but ...
17,"The album features enjoyable, feel-good music,..."
87,The bike rack fits well on an S10 pickup with ...


'Label 0 Summary: "From Potters Field" by Patricia Cornwell is considered one of her weaker novels, mainly due to an unconvincing villain and a lackluster story. While the characters and writing remain strong, the plot fails to deliver excitement, making it a disappointing addition to the Scarpetta series despite good forensic details.\nLabel 1 Summary: "Funny People" is an engaging film about a comedian facing a terminal illness that disrupts his life. It explores his struggle to overcome the illness and rediscover joy through friendships and a romantic relationship.\nLabel 1 Summary: Clay Aiken\'s album Measure of a Man is praised for its impressive collection of potential #1 hits and his breathtaking, pure voice. The reviewer believes it will revolutionize the music industry and highly recommends it to music lovers of all ages.\nLabel 0 Summary: HP removed the Mac OS X driver for the 1022 printer from its website, likely to prevent Mac users from using it as a workaround for the 102

'Label 0 Summary: "From Potters Field" by Patricia Cornwell is considered one of her weaker novels, mainly due to an unconvincing villain and a lackluster story. While the characters and writing remain strong, the plot fails to deliver excitement, making it a disappointing addition to the Scarpetta series despite good forensic details.\nLabel 1 Summary: "Funny People" is an engaging film about a comedian facing a terminal illness that disrupts his life. It explores his struggle to overcome the illness and rediscover joy through friendships and a romantic relationship.\nLabel 1 Summary: Clay Aiken\'s album Measure of a Man is praised for its impressive collection of potential #1 hits and his breathtaking, pure voice. The reviewer believes it will revolutionize the music industry and highly recommends it to music lovers of all ages.\nLabel 0 Summary: HP removed the Mac OS X driver for the 1022 printer from its website, likely to prevent Mac users from using it as a workaround for the 102

In [159]:
from IPython.core.interactiveshell import dis
# Part 4.2 – Implement ask_review_assistant(question: str) -> str
# TODO: implement the chat call using the context and system prompt described above.

def ask_review_assistant(question: str) -> str:
    display(question)
    system_message = {
        "role": "system",
        "content": (
            "You are an assistant that answers questions about a product using ONLY the review summaries "
            "in the provided context. If the context does not contain enough information, say you don't know."
        )
    }

    user_message = {
        "role": "user",
        "content": f"Question: {question}"
    }

    response = client.chat.completions.create(
        model=model,
        messages=[system_message, user_message],
        temperature=0.2,
        max_tokens=250
    )

    display(response)
    answer = response.choices[0].message.content.strip()
    return answer

response=ask_review_assistant(context_string)
display(response)

'Label 0 Summary: "From Potters Field" by Patricia Cornwell is considered one of her weaker novels, mainly due to an unconvincing villain and a lackluster story. While the characters and writing remain strong, the plot fails to deliver excitement, making it a disappointing addition to the Scarpetta series despite good forensic details.\nLabel 1 Summary: "Funny People" is an engaging film about a comedian facing a terminal illness that disrupts his life. It explores his struggle to overcome the illness and rediscover joy through friendships and a romantic relationship.\nLabel 1 Summary: Clay Aiken\'s album Measure of a Man is praised for its impressive collection of potential #1 hits and his breathtaking, pure voice. The reviewer believes it will revolutionize the music industry and highly recommends it to music lovers of all ages.\nLabel 0 Summary: HP removed the Mac OS X driver for the 1022 printer from its website, likely to prevent Mac users from using it as a workaround for the 102

BadRequestError: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}

In [ ]:
# Part 4.2 – Demonstrate the assistant with at least 3 questions
# TODO: Call ask_review_assistant(...) three times with different questions and print the responses.

